# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata_obj = dataset.metadata
print("Dataset title:", metadata_obj.name)
print("Dataset description:", metadata_obj.description)
print("Dataset identifier:", metadata_obj.identifier)


## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema describes the dataset structure. Let's inspect the metadata for available record sets and their fields.

- **Record sets** represent logical tables or groupings of records.
- Each record set and field is uniquely referenced by its `@id`.


In [ ]:
# List all available record sets and their @id
record_sets = [rs['@id'] for rs in metadata_obj.record_sets]
print("Record sets (@id):")
for rid in record_sets:
    print(f"- {rid}")

# For each record set, print field @ids
for rs_meta in metadata_obj.record_sets:
    print("\nRecordSet @id:", rs_meta['@id'])
    print("Fields:")
    if 'field' in rs_meta:
        for field in rs_meta['field']:
            print(f"  - {field['@id']}: {field.get('name', '')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

We'll extract all data for each record set.

In [ ]:
# Define record sets by their @id
record_set_ids = record_sets  # list from the previous step
dataframes = {}

# Extract records into DataFrame for each record set
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"=== {rs_id} ===")
        print("Columns:", dataframes[rs_id].columns.tolist())
        print(dataframes[rs_id].head(), "\n")
    else:
        print(f"No records found for record set {rs_id}")

# Select a record set for further analysis
# If more than one record set, pick the first one
main_record_set_id = record_set_ids[0] if record_set_ids else None


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here we will:
- Filter records on a numeric field using its `@id`.
- Normalize the values.
- Group by another field and display statistics.


In [ ]:
# Using record set and field @id for EDA
if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # List numeric fields, using the 'field' metadata
    rs_meta = [rs for rs in metadata_obj.record_sets if rs['@id'] == main_record_set_id][0]
    numeric_fields = []
    if 'field' in rs_meta:
        for field_meta in rs_meta['field']:
            if field_meta.get('dataType', '').lower() in ['integer', 'float', 'number']:
                numeric_fields.append(field_meta['@id'])

    # Choose the first numeric field for demonstration
    numeric_field_id = numeric_fields[0] if numeric_fields else None

    # Ensure field names match DataFrame columns
    if numeric_field_id and numeric_field_id in df.columns:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = 10
        try:
            filtered_df = df[df[numeric_field_id].astype(float) > threshold]
        except Exception:
            filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
        ) / filtered_df[numeric_field_id].astype(float).std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a categorical field for grouping
        categorical_fields = []
        for field_meta in rs_meta['field']:
            if field_meta.get('dataType', '').lower() == 'text':
                categorical_fields.append(field_meta['@id'])
        group_field_id = categorical_fields[0] if categorical_fields else None

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found or present in DataFrame columns.")
else:
    print("No main record set available for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we show histograms and scatterplots for available numeric and categorical fields from the selected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    if numeric_field_id and numeric_field_id in df.columns:
        # Histogram of numeric field
        plt.figure(figsize=(6,4))
        sns.histplot(df[numeric_field_id].astype(float), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

        # If grouping field is available, plot boxplot
        if group_field_id and group_field_id in df.columns:
            plt.figure(figsize=(8,4))
            sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()
    else:
        print("No numeric field available for visualization.")
else:
    print("No data available for visualization.")


## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset defined by a Croissant schema:
- Loaded metadata and extracted record sets with their unique `@id`s.
- Demonstrated extraction and loading of tabular data using `mlcroissant`.
- Applied simple EDA steps: filtering, normalization, grouping.
- Visualized distributions and relationships between variables.

The dataset provides valuable insights into predictors of knowledge adoption in rangeland management, with demographic, geographic, and logistic regression outputs useful for policy and academic research. Further analysis can focus on specific predictors, bias evaluation, or time series behavior as per the dataset structure.